# WAKIS FD – SIBC $D_{Z_s}$ Diagnose

Gleiche konforme STL-Geometrie für PEC und SIBC; Vergleich der elektrischen DOFs, SIBC-Diagonale und frequenzabhängigen Systemmatrix bei **1 GHz**. Benötigt die neue `solverFIT3D_FD.py` mit `_build_sibc_geometry` und den temporären PEC-artigen SIBC-Volumenmaterial-Workaround. Notebook aus dem Repository starten.

In [1]:
import numpy as np
import pyvista as pv
from pathlib import Path
from scipy.constants import pi
from tempfile import TemporaryDirectory

from wakis import GridFIT3D
from wakis.solverFIT3D_FD import SolverFIT3D_FD

In [2]:
# Identische Testgeometrie wie im bisherigen pytest-Regressionstest.
# Achtung: Diese Geometrie meldet ggf. einige Mehrfachschnitte.
tmp = TemporaryDirectory()
stl_path = Path(tmp.name) / "conductor.stl"

solid = pv.Cube(
    center=(0.08, -0.04, 0.03),
    x_length=0.97,
    y_length=0.85,
    z_length=0.92,
).triangulate()
solid.rotate_z(21.0, inplace=True)
solid.rotate_y(13.0, inplace=True)
solid.save(str(stl_path))

CONFORMAL_ETA = 0.2

# Conservative TD Courant factor for the conformal geometry. The current
# Cartesian CFL estimate does not yet account for shortened conformal metrics.
TD_CFLN = 0.1


def make_solver(material_type):
    material = [np.inf, 1.0, 0.0] if material_type == "pec" else [1.0, 1.0, 5.8e7]
    grid = GridFIT3D(
        xmin=-1.0,
        xmax=1.0,
        ymin=-1.0,
        ymax=1.0,
        zmin=-1.0,
        zmax=1.0,
        Nx=12,
        Ny=12,
        Nz=12,
        stl_solids={"wall": str(stl_path)},
        stl_materials={"wall": material},
        stl_material_types={"wall": material_type},
        geometry_mode="conformal",
        stl_method="interior_points",
        bg_material_type="normal",
        conformal_pec_snap_fraction=CONFORMAL_ETA,
        verbose=0,
    )
    return SolverFIT3D_FD(
        grid=grid,
        use_stl=True,
        use_sibc=False,  # Alter TD-SIBC-Schalter bleibt aus
        bc_low=["PEC"] * 3,
        bc_high=["PEC"] * 3,
        verbose=0,
    )


solver_pec = make_solver("pec")
solver_sibc = make_solver("sibc")

print("PEC  active E DOFs:", len(solver_pec.free_E_idx))
print("SIBC active E DOFs:", len(solver_sibc.free_E_idx))
assert np.array_equal(solver_pec.free_E_idx, solver_sibc.free_E_idx), (
    "DOF-Listen weichen ab"
)

Generating grid with 1728 mesh cells...


/home/hschreib/Documents/Wakis/WAKIS/wakis/gridFIT3D.py:4428: UserWarning: Conformal geometry diagnostics:
138 edges contain geometric crossings that are hidden from endpoint topology and are ignored by the current conformal discretization.
10 edges contain multiple crossing positions. 5 of these are currently used by the conformal discretization; one representative crossing is selected on each edge.
10 edges cross the same STL at more than one distinct position. The geometry on these edges is not fully resolved by the current single-crossing treatment.
  primal x-edge (7, 4, 3): wall
  primal x-edge (7, 9, 3): wall
  primal y-edge (10, 4, 5): wall
  primal z-edge (3, 6, 4): wall
  dual y-edge (2, 6, 4): wall
  ... and 5 more.
  self._warn_interface_diagnostics()


Initializing Electromagnetic solver...
Generating grid with 1728 mesh cells...
Initializing Electromagnetic solver...
PEC  active E DOFs: 4718
SIBC active E DOFs: 4718


In [3]:
import numpy as np

solver = make_solver("pec")

ieps = solver.iDeps.diagonal()
geom = solver.active_conductor_E
boundary = solver.bc_mask_E.toarray().astype(bool)

geometrically_active = geom & boundary

bad = geometrically_active & (ieps == 0.0)

print("Geometrically allowed E DOFs:", np.count_nonzero(geometrically_active))

print("Active E DOFs:", len(solver.free_E_idx))

print("Geometrically allowed DOFs with ieps=0:", np.count_nonzero(bad))

print(
    "Geometric DOFs lost through material masking:",
    np.count_nonzero(geometrically_active) - len(solver.free_E_idx),
)

Generating grid with 1728 mesh cells...
Initializing Electromagnetic solver...
Geometrically allowed E DOFs: 4718
Active E DOFs: 4718
Geometrically allowed DOFs with ieps=0: 0
Geometric DOFs lost through material masking: 0


In [4]:
import numpy as np

s = solver_pec
bad = (
    s.active_conductor_E
    & s.bc_mask_E.toarray().astype(bool)
    & (s.iDeps.diagonal() == 0.0)
)

print("Problematic E DOFs:", np.count_nonzero(bad))

for component, direction in enumerate(("x", "y", "z")):
    sl = slice(component * s.N, (component + 1) * s.N)
    b = bad[sl].reshape((s.Nx, s.Ny, s.Nz))

    start, end = s.grid._get_edge_region_ids("primal", direction)
    start = s._edge_array_to_field_component(start, direction)
    end = s._edge_array_to_field_component(end, direction)

    normal_ids = [
        rid for rid, kind in s.region_material_types.items() if kind == "normal"
    ]
    conductor_ids = [
        rid for rid, kind in s.region_material_types.items() if kind in ("pec", "sibc")
    ]

    sn = np.isin(start, normal_ids)
    en = np.isin(end, normal_ids)
    sc = np.isin(start, conductor_ids)
    ec = np.isin(end, conductor_ids)

    same_normal = sn & en & (start == end)
    different_normal = sn & en & (start != end)
    crossing = (sn & ec) | (sc & en)
    conductor_conductor = sc & ec

    data = s.conformal_line_data["primal"][direction]
    supported = s._edge_array_to_field_component(data["supported"], direction).astype(
        bool
    )

    print(f"\n{direction}-edges: {np.count_nonzero(b)}")
    print("  Same normal region:", np.count_nonzero(b & same_normal))
    print("  Different normal regions:", np.count_nonzero(b & different_normal))
    print("  Normal-conductor:", np.count_nonzero(b & crossing))
    print("  Conductor-conductor:", np.count_nonzero(b & conductor_conductor))
    print("  Supported conformal intersection:", np.count_nonzero(b & supported))
    print("  Unsupported/no resolved intersection:", np.count_nonzero(b & ~supported))

Problematic E DOFs: 0

x-edges: 0
  Same normal region: 0
  Different normal regions: 0
  Normal-conductor: 0
  Conductor-conductor: 0
  Supported conformal intersection: 0
  Unsupported/no resolved intersection: 0

y-edges: 0
  Same normal region: 0
  Different normal regions: 0
  Normal-conductor: 0
  Conductor-conductor: 0
  Supported conformal intersection: 0
  Unsupported/no resolved intersection: 0

z-edges: 0
  Same normal region: 0
  Different normal regions: 0
  Normal-conductor: 0
  Conductor-conductor: 0
  Supported conformal intersection: 0
  Unsupported/no resolved intersection: 0


## 1. Unabhängige Prüfung der SIBC-Flächen

Wir rekonstruieren `g = l_s sin(theta) / A_field` direkt aus den gespeicherten primalen STL-Schnittsegmenten. Danach vergleichen wir jede Fläche über die **FIT-Indexverschiebung** mit `sibc_g`. Die Prüfung beruht auf denselben Geometriedaten wie die Implementierung; sie testet Konsistenz und Mapping, noch **nicht** die geometrische Richtigkeit der STL-Intersektionen oder die physikalische Gültigkeit der skalaren Näherung.

In [8]:
from collections import Counter

s = solver_sibc
N = s.N
axis_map = {"x": 0, "y": 1, "z": 2}
rows = []

for direction, axis in axis_map.items():
    geom = s.grid.interface_face_data["primal"][direction]
    face_info = s.conformal_face_data["primal"][direction]
    expected_g = np.zeros(geom["resolved"].shape, dtype=float)
    expected_sigma = np.zeros_like(expected_g)
    expected_mu_wall = np.zeros_like(expected_g)
    expected_mu_field = np.ones_like(expected_g)

    for idx, segment in geom["segments"].items():
        region_id = int(segment["surface_id"])
        if s.region_material_types[region_id] != "sibc":
            continue
        assert bool(geom["resolved"][idx]), (direction, idx, "unresolved")
        assert bool(face_info["normal_sibc"][idx]), (direction, idx, "not normal/SIBC")
        field_ids = face_info["normal_region_ids"][idx]
        wall_ids = face_info["conductor_region_ids"][idx]
        assert len(field_ids) == len(wall_ids) == 1 and wall_ids[0] == region_id

        area = float(geom["region_areas"][idx][field_ids[0]])
        length = float(segment["length"])
        normal = np.asarray(segment["normal"], dtype=float)
        norm = np.linalg.norm(normal)
        assert np.isfinite(area) and area > 0
        assert np.isfinite(length) and length > 0
        assert np.isfinite(norm) and norm > 0
        normal = normal / norm
        sin_theta = np.sqrt(max(0.0, 1.0 - normal[axis] ** 2))
        assert sin_theta > 1e-10, (direction, idx, "degenerate angle")

        field_mat = s._get_region_material_properties(field_ids[0])
        wall_mat = s._get_region_material_properties(region_id)
        assert wall_mat[2] > 0 and wall_mat[1] > 0 and field_mat[1] > 0

        g = length * sin_theta / area
        expected_g[idx] = g
        expected_sigma[idx] = wall_mat[2]
        expected_mu_wall[idx] = wall_mat[1]
        expected_mu_field[idx] = field_mat[1]

        # Hx(i,j,k) belongs to primal x-face (i+1,j,k), etc.
        shifted = list(idx)
        shifted[axis] -= 1
        assert shifted[axis] >= 0, (direction, idx, "low-side face not stored")
        stored_local = np.ravel_multi_index(tuple(shifted), (s.Nx, s.Ny, s.Nz))
        stored_global = axis * N + stored_local
        rows.append(
            dict(
                direction=direction,
                primal_index=idx,
                H_dof=stored_global,
                length=length,
                area=area,
                sin_theta=sin_theta,
                g=g,
            )
        )

    start, end = axis * N, (axis + 1) * N
    for label, face_data, actual in (
        ("g", expected_g, s.sibc_g[start:end]),
        ("sigma", expected_sigma, s.sibc_sigma[start:end]),
        ("mu_wall", expected_mu_wall, s.sibc_mu_wall[start:end]),
        ("mu_field", expected_mu_field, s.sibc_mu_field[start:end]),
    ):
        mapped = s._primal_face_array_to_dual_edge_component(
            face_data, direction
        ).ravel(order="F")
        np.testing.assert_allclose(
            mapped,
            actual,
            rtol=1e-12,
            atol=1e-12,
            err_msg=f"{direction}: {label} mapping differs",
        )

print("Primal SIBC segments:", len(rows))
print("Unique magnetic H DOFs:", len({r["H_dof"] for r in rows}))
print("Nonzero sibc_g entries:", np.count_nonzero(s.sibc_g > 0))
assert len(rows) == len({r["H_dof"] for r in rows}), (
    "Multiple segments map to same DOF: potential overwrite!"
)
assert len(rows) == np.count_nonzero(s.sibc_g > 0), "Missing or additional SIBC entries"
print("Segments by normal direction:", dict(Counter(r["direction"] for r in rows)))

Primal SIBC segments: 410
Unique magnetic H DOFs: 410
Nonzero sibc_g entries: 410
Segments by normal direction: {'x': 129, 'y': 138, 'z': 143}


## 2. Verteilung der geometrischen Faktoren

Gesondert betrachten wir extrem kleine aktive Flächen und Winkel: Beide können die Stabilität der Approximation beeinflussen. Auffällige Werte sind zunächst Diagnosen, keine automatischen Fehler.

In [9]:
import pandas as pd

df = pd.DataFrame(rows)
print(df[["length", "area", "sin_theta", "g"]].describe().to_string())
print("\nGrößte g-Faktoren:")
print(df.nlargest(10, "g").to_string(index=False))
print("\nKleinste sin(theta):")
print(df.nsmallest(10, "sin_theta").to_string(index=False))

# Prüfen, ob ausschließlich aktive SIBC-Magnetfreiheitsgrade modifiziert werden.
for f in (1e8, 1e9, 1e10):
    D = s._build_sibc_diagonal(2 * np.pi * f)
    altered = ~np.isclose(D, 1.0, rtol=0.0, atol=1e-14)
    expected = s.sibc_g > 0
    assert np.array_equal(altered, expected), f"Incorrect D_Zs support at {f} Hz"
    print(f"f={f / 1e9:.2f} GHz: {np.count_nonzero(altered)} modified D_Zs entries")

           length        area   sin_theta           g
count  410.000000  410.000000  410.000000  410.000000
mean     0.149243    0.018372    0.905896   12.440885
std      0.047733    0.008257    0.179920   13.152521
min      0.002978    0.001455    0.224951    0.081961
25%      0.158208    0.011387    0.933580    4.804886
50%      0.167552    0.021069    0.974370    7.689045
75%      0.172881    0.025692    0.996745   13.169858
max      0.225734    0.027776    1.000000   62.075751

Größte g-Faktoren:
direction primal_index  H_dof   length     area  sin_theta         g
        z    (5, 3, 9)   4220 0.092367 0.001455   0.977699 62.075751
        z    (9, 6, 8)   4831 0.093454 0.001489   0.977699 61.354140
        z    (6, 3, 8)   4363 0.101254 0.001682   0.996745 60.001938
        z    (8, 4, 5)   4660 0.104633 0.001796   0.996745 58.064066
        y    (6, 4, 8)   2636 0.158208 0.002743   1.000000 57.675340
        y    (6, 5, 8)   2648 0.158208 0.002743   1.000000 57.675340
        y  

## 3. Nächste Validierungsstufe

Die bisherige gedrehte, geschlossene STL-Testgeometrie weist **zehn bereits gemeldete Mehrfach-/Hidden-Crossing-Kanten** auf. Darum bestätigt dieses Notebook zunächst nur die **interne Konsistenz** von Flächen-Daten, Index-Mapping und `D_Zs`-Support. Für einen unabhängigen Geometrietest sollten wir als Nächstes einen einfach geschnittenen planaren Leiter erzeugen und die analytischen Schnitte gegen die STL-Segmente prüfen. Erst danach folgt die physikalische Resonatorvalidierung (Resonanzverschiebung, Dämpfung und Passivität).

In [10]:
import numpy as np

C = solver_sibc.C_red.tocsr()
g = solver_sibc.sibc_g

active_sibc = g > 0

row_nnz = np.diff(C.indptr)

effective_sibc = active_sibc & (row_nnz > 0)
inactive_sibc = active_sibc & (row_nnz == 0)

print("SIBC magnetic DOFs:", np.count_nonzero(active_sibc))
print("Effective SIBC DOFs:", np.count_nonzero(effective_sibc))
print("Inactive SIBC DOFs:", np.count_nonzero(inactive_sibc))

if np.any(inactive_sibc):
    print("Largest inactive g:", np.max(g[inactive_sibc]))

SIBC magnetic DOFs: 410
Effective SIBC DOFs: 362
Inactive SIBC DOFs: 48
Largest inactive g: 62.07575126465609


In [11]:
import numpy as np

g = solver_sibc.sibc_g
C = solver_sibc.C_red.tocsr()

active = g > 0
effective = active & (np.diff(C.indptr) > 0)
inactive = active & (np.diff(C.indptr) == 0)

# iA is a WAKIS Field object, not a scipy sparse matrix.
inverse_area = solver_sibc.iA.toarray()

# Effective primal face area
area = np.divide(
    1.0,
    inverse_area,
    out=np.zeros_like(inverse_area, dtype=float),
    where=inverse_area != 0,
)

for name, mask in [
    ("Effective", effective),
    ("Inactive", inactive),
]:
    values = area[mask]

    print(f"\n{name} SIBC faces: {len(values)}")
    print("Min area:", values.min())
    print("Median area:", np.median(values))
    print("Max area:", values.max())

    print("Median g:", np.median(g[mask]))
    print("Max g:", np.max(g[mask]))


Effective SIBC faces: 362
Min area: 0.0016820208843465053
Median area: 0.024733301666047848
Max area: 0.027777777777777804
Median g: 7.470606738891094
Max g: 61.35414004316333

Inactive SIBC faces: 48
Min area: 0.027777777777777728
Median area: 0.027777777777777776
Max area: 0.027777777777777804
Median g: 14.857214147609255
Max g: 62.07575126465609


In [12]:
import numpy as np

solver = solver_sibc

# Full discrete curl matrix.
C_full = solver.C.tocsr()

# Existing reduced curl matrix.
C_material = solver.C_red.tocsr()

# Geometric mask and outer electrical boundary conditions.
active_geometry = solver.active_conductor_E & solver.bc_mask_E.toarray().astype(bool)

free_geometry = np.flatnonzero(active_geometry)

C_geometry = C_full[:, free_geometry].tocsr()

# Magnetic DOFs carrying SIBC corrections.
sibc_mask = solver.sibc_g > 0

material_row_active = np.diff(C_material.indptr) > 0
geometry_row_active = np.diff(C_geometry.indptr) > 0

# SIBC faces currently inactive but restored by geometric masking.
restored = sibc_mask & ~material_row_active & geometry_row_active

# SIBC faces inactive under both masks.
still_inactive = sibc_mask & ~material_row_active & ~geometry_row_active

print("SIBC magnetic DOFs:", np.count_nonzero(sibc_mask))

print(
    "Active with current material mask:",
    np.count_nonzero(sibc_mask & material_row_active),
)

print("Active with geometric mask:", np.count_nonzero(sibc_mask & geometry_row_active))

print("Restored by geometric mask:", np.count_nonzero(restored))

print("Still inactive with geometric mask:", np.count_nonzero(still_inactive))

print(
    "Additional geometric electric DOFs:", len(free_geometry) - len(solver.free_E_idx)
)

SIBC magnetic DOFs: 410
Active with current material mask: 362
Active with geometric mask: 362
Restored by geometric mask: 0
Still inactive with geometric mask: 48
Additional geometric electric DOFs: 0


In [13]:
C = solver_sibc.C_red.tocsr()
active = solver_sibc.sibc_g > 0
effective = active & (np.diff(C.indptr) > 0)

print("SIBC magnetic DOFs:", np.count_nonzero(active))
print("Effective SIBC DOFs:", np.count_nonzero(effective))
print("Inactive SIBC DOFs:", np.count_nonzero(active & ~effective))

SIBC magnetic DOFs: 410
Effective SIBC DOFs: 362
Inactive SIBC DOFs: 48


In [14]:
import numpy as np

s = solver_sibc

C_full = s.C.tocsr()
C_red = s.C_red.tocsr()

sibc = s.sibc_g > 0

full_nnz = np.diff(C_full.indptr)
red_nnz = np.diff(C_red.indptr)

inactive = sibc & (red_nnz == 0)

# Already zero in the full curl matrix
zero_full = inactive & (full_nnz == 0)

# Nonzero in full curl, but zero after E-DOF reduction
lost_reduction = inactive & (full_nnz > 0)

print("Inactive SIBC faces:", np.count_nonzero(inactive))

print("Already zero in full C:", np.count_nonzero(zero_full))

print("Lost through electric DOF reduction:", np.count_nonzero(lost_reduction))

# Analyze which electric DOFs belong to the removed rows
if np.any(lost_reduction):
    rows = np.flatnonzero(lost_reduction)

    involved_e = np.unique(
        np.concatenate(
            [C_full.indices[C_full.indptr[r] : C_full.indptr[r + 1]] for r in rows]
        )
    )

    print("Distinct E DOFs on these face boundaries:", len(involved_e))

    print(
        "Excluded by geometric conductor mask:",
        np.count_nonzero(~s.active_conductor_E[involved_e]),
    )

    print(
        "Excluded by outer boundary conditions:",
        np.count_nonzero(~s.bc_mask_E.toarray().astype(bool)[involved_e]),
    )

Inactive SIBC faces: 48
Already zero in full C: 0
Lost through electric DOF reduction: 48
Distinct E DOFs on these face boundaries: 150
Excluded by geometric conductor mask: 150
Excluded by outer boundary conditions: 0


In [15]:
import numpy as np

s = solver_sibc

C = s.C_red.tocsr()
inactive = (s.sibc_g > 0) & (np.diff(C.indptr) == 0)

for direction, axis in zip(("x", "y", "z"), range(3)):
    start = axis * s.N
    end = start + s.N

    mask = inactive[start:end]

    print(f"{direction}: {np.count_nonzero(mask)} inactive SIBC DOFs")

x: 23 inactive SIBC DOFs
y: 14 inactive SIBC DOFs
z: 11 inactive SIBC DOFs


In [17]:
C_full = s.C.tocsr()

geom = s.active_conductor_E
bc = s.bc_mask_E.toarray().astype(bool)

counts = {
    "STL_only": 0,
    "BC_only": 0,
    "both": 0,
    "neither": 0,
}

for h_idx in np.flatnonzero(inactive):
    row = C_full.getrow(h_idx)
    e_indices = row.indices

    for e_idx in e_indices:
        conductor_removed = not geom[e_idx]
        boundary_removed = not bc[e_idx]

        if conductor_removed and boundary_removed:
            counts["both"] += 1
        elif conductor_removed:
            counts["STL_only"] += 1
        elif boundary_removed:
            counts["BC_only"] += 1
        else:
            counts["neither"] += 1

print(counts)

{'STL_only': 192, 'BC_only': 0, 'both': 0, 'neither': 0}
